<!-- nav -->
[← 6 · Drawing values](06_Drawing_Values.ipynb) · [Home](../../README.md) · [Patterns design →](../../docs/PATTERNS.md)

# 7 · Generating test data

**The problem.** Tests need data that looks like production's mix, reproducibly: the same seed must give the same
contacts tomorrow, in CI, and in the Python test suite. `Generators.Generate` builds new matches of a pattern.

In [1]:
import { Expressions as E } from "@mbse/expressions";
import { Distributions, Generators, Predicates, Queries, Validators } from "@mbse/patterns";
import { Stores } from "@mbse/schemas/Framework";
import { Contact, Directory, Phone, book, listed } from "./toolkit.ts";

const person = E.variable("person");
const HasName = new Predicates.OfPredicate.Builder().name("HasName").symbols({ person: Contact })
  .parameters((x) => x.name("name")).requires(person.name.eq(E.variable("name"))).create();
const aged = (distribution: any) => distribution.symbol("age").requires((age: any) => person.age.eq(age)).create();
const People = new Predicates.OfPredicate.Builder().name("People").symbols({ person: Contact }).requires(
  new Distributions.Choices.Builder().arms(
    (a) => a.weight(3).requires(HasName.call(person, "senior"), person.age.ge(65n),
      aged(new Distributions.Normal.Builder().mean(70n).deviation(8n).rounded())),
    (a) => a.weight(7).requires(HasName.call(person, "adult"), aged(new Distributions.Uniform.Builder().low(18n).high(64n))),
  ).decreasing().create()).create();

## Step 1: generating

Each step chooses an arm by weight, draws the values the arm's distributions give, sets the properties the equalities
say, and builds a contact with the store's builders:

In [2]:
const store = book();
const generated = Generators.Generate(store, People, new Stores.PCG32(42n));
const people = Array.from({ length: 1000 }, () => (generated.next().value as any).person);
console.log(people.slice(0, 6).map((x) => [x.name, x.age]));

[
  [ "adult", 51n ],
  [ "senior", 66n ],
  [ "adult", 48n ],
  [ "adult", 27n ],
  [ "adult", 48n ],
  [ "adult", 24n ]
]


In [3]:
const seniors = people.filter((x) => x.name === "senior").map((x) => Number(x.age));
console.log(seniors.length, people.length - seniors.length,
  Math.round(seniors.reduce((a, b) => a + b, 0) / seniors.length * 10) / 10, Math.min(...seniors));

283 717 73.6 65


About three in ten are seniors, their ages near 70 and never under 65. A senior's normal can draw 60; then the arm
doesn't hold, and the step draws again. That's **rejection**: the generator counts it, so you can see what a constraint
costs:

In [4]:
console.log(generated.steps, generated.rejected);

1000 103


## Step 2: the same seed, the same data

Each step draws from its own stream, split from the seed, so the tenth contact doesn't depend on how many came before
it, and the same seed gives the same contacts. The Python generator gives these exact ones too: both implement the
sampling, and even `log` and `exp`, with the same arithmetic.

In [5]:
const again = Generators.Generate(book(), People, new Stores.PCG32(42n));
console.log(people.slice(0, 6).every((x) => (again.next().value as any).person.age === x.age));

true


## Step 3: generated data is ordinary data

The contacts are transient until you list them. Then they validate and query like any other data:

In [6]:
listed(store, ...people.slice(0, 20));
const problems = Validators.Validate(store, [People]).Reachable(Directory, store.singleton("book.Directory"));
const found = [...new Queries.Scan(store).select(People)].filter((m: any) => m.person.name === "senior");
console.log(problems, found.length);

[] 4


## Step 4: what a generator can't do

A generator sets what equalities say and draws what distributions give. It doesn't invent links: a pattern that
requires a contact to have a phone can't be built from nothing, and says so:

In [7]:
const p = E.variable("p");
const Phoned = new Predicates.OfPredicate.Builder().symbols({ person: Contact }).requires(Predicates.Exists(
  (q) => q.symbols({ p: Phone }).requires(Predicates.Contains(person.phones, (e) => e.phone.eq(p))))).create();
try {
  Generators.Generate(book(), Phoned, new Stores.PCG32(1n)).next();
} catch (error) {
  console.log(String(error));
}

ValueError: the predicate cannot be generated from its equalities and choices: what they build does not satisfy it


## Why it works this way

- **Generate from the constraint you check with.** The pattern is a predicate, so what you generate passes your
  validators by construction: the generator checks every match before it yields it.
- **Rejection is honest.** Truncating a normal by redrawing gives exactly the normal restricted to the arm, and the
  count of rejections tells you when a constraint makes generation expensive.
- **Reproducible across languages.** A seed is a contract: the same contacts in TypeScript and Python, today and
  later.

<!-- nav -->
[← 6 · Drawing values](06_Drawing_Values.ipynb) · [Home](../../README.md) · [Patterns design →](../../docs/PATTERNS.md)